# Prepare Tahoe↔L1000 DEG Benchmark Inputs

This notebook builds the common-gene, shared-compound benchmark inputs used by the DEG and LFC transfer notebooks.

It does four things:

1. discovers the filtered L1000 donors in `data/l1000_narrowed_to_tahoe`
2. narrows Tahoe to overlapping cell lines and overlapping compounds
3. recomputes adjusted p-values on the exact same shared gene set across Tahoe and L1000
4. writes harmonized `.h5ad` files and gene-subset tables for downstream evaluation

The adjusted p-values here are recomputed on the **same common gene universe** so DEG labels are comparable across datasets.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import anndata as ad
import numpy as np
import pandas as pd
from IPython.display import display

from op3_analysis.tahoe_l1000_deg_benchmark import (
    DEFAULT_LABEL_CONFIGS,
    common_gene_set,
    concordant_gene_table,
    discover_filtered_l1000_paths,
    discover_tahoe_overlap_cell_types,
    donor_compound_set,
    prepare_filtered_l1000_adata,
    prepare_tahoe_overlap_adata,
    responsive_gene_table,
)


In [ ]:
FILTERED_DIR = PROJECT_ROOT / "data" / "l1000_narrowed_to_tahoe"
PREP_DIR = PROJECT_ROOT / "data" / "tahoe_l1000_benchmark_prep"
GENE_SUBSET_DIR = PREP_DIR / "gene_subsets"
PREP_DIR.mkdir(parents=True, exist_ok=True)
GENE_SUBSET_DIR.mkdir(parents=True, exist_ok=True)

l1000_paths = discover_filtered_l1000_paths(FILTERED_DIR)
tahoe_overlap_cell_types = discover_tahoe_overlap_cell_types(l1000_paths)
shared_genes = common_gene_set(l1000_paths, tahoe_overlap_cell_types)
shared_compounds = donor_compound_set(l1000_paths)

print("Filtered donor datasets:", sorted(l1000_paths))
print("Overlapping Tahoe cell types:", tahoe_overlap_cell_types)
print("Shared compounds:", len(shared_compounds))
print("Shared genes:", len(shared_genes))


In [ ]:
prepared_paths = {
    "tahoe": PREP_DIR / "tahoe_shared_compounds_shared_genes.h5ad",
    **{name: PREP_DIR / f"{name}_shared_genes.h5ad" for name in l1000_paths},
}

if all(path.exists() for path in prepared_paths.values()):
    tahoe_adata = ad.read_h5ad(prepared_paths["tahoe"])
    donor_adatas = {name: ad.read_h5ad(path) for name, path in prepared_paths.items() if name != "tahoe"}
else:
    donor_adatas = {
        name: prepare_filtered_l1000_adata(
            path=path,
            dataset_name=name,
            common_genes=shared_genes,
            label_configs=DEFAULT_LABEL_CONFIGS,
        )
        for name, path in l1000_paths.items()
    }
    tahoe_adata = prepare_tahoe_overlap_adata(
        l1000_paths=l1000_paths,
        output_cell_types=tahoe_overlap_cell_types,
        label_configs=DEFAULT_LABEL_CONFIGS,
    )
    tahoe_adata.write_h5ad(prepared_paths["tahoe"])
    for name, adata in donor_adatas.items():
        adata.write_h5ad(prepared_paths[name])

dataset_adatas = {"tahoe": tahoe_adata, **donor_adatas}
display(
    pd.DataFrame(
        {
            "dataset_name": list(dataset_adatas),
            "n_obs": [adata.n_obs for adata in dataset_adatas.values()],
            "n_vars": [adata.n_vars for adata in dataset_adatas.values()],
            "n_cell_types": [adata.obs['cell_type'].astype(str).nunique() for adata in dataset_adatas.values()],
            "n_compounds": [adata.obs['pubchem_cid'].astype(str).nunique() for adata in dataset_adatas.values()],
            "output_path": [str(prepared_paths[name]) for name in dataset_adatas],
        }
    )
)


In [ ]:
label_config_df = pd.DataFrame(tahoe_adata.uns.get("label_configs", []))
display(label_config_df)

summary_rows = []
for dataset_name, adata in dataset_adatas.items():
    for config in label_config_df.to_dict("records"):
        labels = np.asarray(adata.layers[config["name"]], dtype=np.int8)
        summary_rows.append(
            {
                "dataset_name": dataset_name,
                "label_layer": config["name"],
                "fdr_threshold": config["fdr_threshold"],
                "min_abs_logfc": config["min_abs_logfc"],
                "n_up": int((labels > 0).sum()),
                "n_down": int((labels < 0).sum()),
                "n_nonzero": int((labels != 0).sum()),
                "nonzero_fraction": float((labels != 0).mean()),
            }
        )
label_summary = pd.DataFrame(summary_rows)
label_summary.to_csv(PREP_DIR / "label_summary.csv", index=False)
display(label_summary.sort_values(["label_layer", "dataset_name"]).reset_index(drop=True))


In [ ]:
STRICT_LAYER = "deg_fdr005_lfc025"
responsive = responsive_gene_table(dataset_adatas, label_layer=STRICT_LAYER)
responsive.to_csv(GENE_SUBSET_DIR / "responsive_gene_stats.csv", index=False)

shared_all = pd.DataFrame({"gene_id": tahoe_adata.var_names.to_numpy(dtype=str)})
shared_all.to_csv(GENE_SUBSET_DIR / "shared_all_genes.csv", index=False)

responsive_wide = (
    responsive.pivot_table(index="gene_id", columns="dataset_name", values="n_nonzero", fill_value=0)
    .reset_index()
)
donor_columns = [col for col in responsive_wide.columns if col not in {"gene_id", "tahoe"}]
responsive_wide["l1000_total_nonzero"] = responsive_wide[donor_columns].sum(axis=1)
frequent_responsive = responsive_wide.loc[
    (responsive_wide.get("tahoe", 0) >= 5) & (responsive_wide["l1000_total_nonzero"] >= 10),
    ["gene_id", "tahoe", "l1000_total_nonzero"],
].sort_values(["tahoe", "l1000_total_nonzero", "gene_id"], ascending=[False, False, True])
frequent_responsive.to_csv(GENE_SUBSET_DIR / "frequent_responsive_genes.csv", index=False)

concordant = concordant_gene_table(
    tahoe_adata=tahoe_adata,
    donor_adatas=donor_adatas,
    label_layer=STRICT_LAYER,
    min_support=10,
)
concordant.to_csv(GENE_SUBSET_DIR / "concordant_gene_scores.csv", index=False)
concordant_subset = concordant.loc[
    (concordant["sign_agree_fraction"] >= 0.60) & (concordant["n_nonzero_both"] >= 5)
].copy()
concordant_subset.to_csv(GENE_SUBSET_DIR / "concordant_genes.csv", index=False)

subset_manifest = pd.DataFrame(
    [
        {"subset_name": "shared_all", "n_genes": len(shared_all), "path": str(GENE_SUBSET_DIR / 'shared_all_genes.csv')},
        {"subset_name": "frequent_responsive", "n_genes": len(frequent_responsive), "path": str(GENE_SUBSET_DIR / 'frequent_responsive_genes.csv')},
        {"subset_name": "concordant", "n_genes": len(concordant_subset), "path": str(GENE_SUBSET_DIR / 'concordant_genes.csv')},
    ]
)
subset_manifest.to_csv(GENE_SUBSET_DIR / "subset_manifest.csv", index=False)

display(subset_manifest)
display(concordant.head(20))


## Outputs

This notebook writes:

- harmonized Tahoe and L1000 `.h5ad` files under `data/tahoe_l1000_benchmark_prep`
- `label_summary.csv`
- gene subset tables under `data/tahoe_l1000_benchmark_prep/gene_subsets`

The downstream DEG and LFC benchmark notebooks assume these outputs exist.